In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as st
from sklearn.feature_selection import r_regression, mutual_info_regression

In [2]:
df = pd.read_excel("proc.xlsx", sheet_name="Sheet1")

In [3]:
df.columns = df.columns.str.replace(" ", "_").str.lower()

In [4]:
df.head()

,age,gender,region,marital_status,number_of_dependants,bmi_category,smoking_status,employment_status,income_level,income_lakhs,medical_history,insurance_plan,annual_premium_amount,diabetes,heart_disease,high_blood_pressure,no_disease,thyroid
0,25,Male,Southeast,Unmarried,0,Normal,No Smoking,Freelancer,10L - 25L,15,No Disease,Bronze,5684,0,0,0,1,0
1,20,Male,Southeast,Unmarried,2,Overweight,No Smoking,Freelancer,10L - 25L,14,No Disease,Bronze,5712,0,0,0,1,0
2,19,Male,Southwest,Unmarried,0,Normal,No Smoking,Freelancer,<10L,8,No Disease,Bronze,4097,0,0,0,1,0
3,23,Female,Southeast,Unmarried,0,Normal,No Smoking,Freelancer,25L - 40L,29,No Disease,Bronze,7803,0,0,0,1,0
4,18,Female,Southeast,Unmarried,1,Normal,Occasional,Freelancer,<10L,8,Diabetes & High blood pressure,Bronze,8111,1,0,1,0,0


In [6]:
df['no_disease'] = df['no_disease'].map({1: 0, 0:1})

In [7]:
df = df.rename(columns={"no_disease": "have_disease"})

In [8]:
df

,age,gender,region,marital_status,number_of_dependants,bmi_category,smoking_status,employment_status,income_level,income_lakhs,medical_history,insurance_plan,annual_premium_amount,diabetes,heart_disease,high_blood_pressure,have_disease,thyroid
0,25,Male,Southeast,Unmarried,0,Normal,No Smoking,Freelancer,10L - 25L,15,No Disease,Bronze,5684,0,0,0,1,0
1,20,Male,Southeast,Unmarried,2,Overweight,No Smoking,Freelancer,10L - 25L,14,No Disease,Bronze,5712,0,0,0,1,0
2,19,Male,Southwest,Unmarried,0,Normal,No Smoking,Freelancer,<10L,8,No Disease,Bronze,4097,0,0,0,1,0
3,23,Female,Southeast,Unmarried,0,Normal,No Smoking,Freelancer,25L - 40L,29,No Disease,Bronze,7803,0,0,0,1,0
4,18,Female,Southeast,Unmarried,1,Normal,Occasional,Freelancer,<10L,8,Diabetes & High blood pressure,Bronze,8111,1,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,64,Female,Southwest,Unmarried,0,Normal,No Smoking,Self-Employed,<10L,3,No Disease,Silver,20380,0,0,0,1,0
49996,54,Male,Southwest,Married,3,Obesity,Regular,Self-Employed,25L - 40L,28,Diabetes,Silver,25447,1,0,0,0,0
49997,42,Male,Southwest,Married,3,Normal,Occasional,Self-Employed,25L - 40L,33,Diabetes,Silver,16102,1,0,0,0,0
49998,55,Male,Southwest,Married,2,Overweight,Regular,Self-Employed,10L - 25L,16,Diabetes,Silver,23518,1,0,0,0,0


In [9]:
df = df.drop("medical_history", axis = 1)
df.head()

,age,gender,region,marital_status,number_of_dependants,bmi_category,smoking_status,employment_status,income_level,income_lakhs,insurance_plan,annual_premium_amount,diabetes,heart_disease,high_blood_pressure,have_disease,thyroid
0,25,Male,Southeast,Unmarried,0,Normal,No Smoking,Freelancer,10L - 25L,15,Bronze,5684,0,0,0,1,0
1,20,Male,Southeast,Unmarried,2,Overweight,No Smoking,Freelancer,10L - 25L,14,Bronze,5712,0,0,0,1,0
2,19,Male,Southwest,Unmarried,0,Normal,No Smoking,Freelancer,<10L,8,Bronze,4097,0,0,0,1,0
3,23,Female,Southeast,Unmarried,0,Normal,No Smoking,Freelancer,25L - 40L,29,Bronze,7803,0,0,0,1,0
4,18,Female,Southeast,Unmarried,1,Normal,Occasional,Freelancer,<10L,8,Bronze,8111,1,0,1,0,0


## Feature selection

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 17 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   age                    50000 non-null  int64 
 1   gender                 50000 non-null  object
 2   region                 50000 non-null  object
 3   marital_status         50000 non-null  object
 4   number_of_dependants   50000 non-null  int64 
 5   bmi_category           50000 non-null  object
 6   smoking_status         50000 non-null  object
 7   employment_status      50000 non-null  object
 8   income_level           50000 non-null  object
 9   income_lakhs           50000 non-null  int64 
 10  insurance_plan         50000 non-null  object
 11  annual_premium_amount  50000 non-null  int64 
 12  diabetes               50000 non-null  int64 
 13  heart_disease          50000 non-null  int64 
 14  high_blood_pressure    50000 non-null  int64 
 15  have_disease       

In [12]:
# lets combine all disease columns to -> no of disease
df['n_disease'] = df['diabetes'] + df['heart_disease'] + df['high_blood_pressure'] + df['thyroid']

In [13]:
df = df.drop(['diabetes', 'heart_disease', 'high_blood_pressure', 'thyroid', 'have_disease'], axis = 1)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   age                    50000 non-null  int64 
 1   gender                 50000 non-null  object
 2   region                 50000 non-null  object
 3   marital_status         50000 non-null  object
 4   number_of_dependants   50000 non-null  int64 
 5   bmi_category           50000 non-null  object
 6   smoking_status         50000 non-null  object
 7   employment_status      50000 non-null  object
 8   income_level           50000 non-null  object
 9   income_lakhs           50000 non-null  int64 
 10  insurance_plan         50000 non-null  object
 11  annual_premium_amount  50000 non-null  int64 
 12  n_disease              50000 non-null  int64 
dtypes: int64(5), object(8)
memory usage: 5.0+ MB


In [14]:
cat = df.select_dtypes("object").columns
cat

Index(['gender', 'region', 'marital_status', 'bmi_category', 'smoking_status',
       'employment_status', 'income_level', 'insurance_plan'],
      dtype='object')

let's encode categorical features for feature selection in r_regression

In [16]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

In [ ]:
oe = OrdinalEncoder(
    categories=[
        ['Female', 'Male'], #gender
        ['Northeast', 'Northwest', 'Southeast', 'Southwest'], #region     
        ['Unmarried', 'Married'], #marital status      
        ['Underweight', 'Normal', 'Overweight', 'Obesity'],  #BMI category     
        ['No Smoking', 'Occasional', 'Regular'],  #smoking status
        ['Freelancer', 'Self-Employed', 'Salaried'],  #employment status      
        ['<10L', '10L - 25L', '25L - 40L','> 40L'],   #income level    
        ['Bronze', 'Silver', 'Gold']  #insurance plan     
    ]
)
encoded = oe.fit_transform(df[cat])
encoded

array([[1., 2., 0., ..., 0., 1., 0.],
       [1., 2., 0., ..., 0., 1., 0.],
       [1., 3., 0., ..., 0., 0., 0.],
       ...,
       [1., 3., 1., ..., 1., 2., 1.],
       [1., 3., 1., ..., 1., 1., 1.],
       [0., 3., 1., ..., 1., 0., 1.]])

In [18]:
oe.categories_

[array(['Female', 'Male'], dtype=object),
 array(['Northeast', 'Northwest', 'Southeast', 'Southwest'], dtype=object),
 array(['Unmarried', 'Married'], dtype=object),
 array(['Underweight', 'Normal', 'Overweight', 'Obesity'], dtype=object),
 array(['No Smoking', 'Occasional', 'Regular'], dtype=object),
 array(['Freelancer', 'Self-Employed', 'Salaried'], dtype=object),
 array(['<10L', '10L - 25L', '25L - 40L', '> 40L'], dtype=object),
 array(['Bronze', 'Silver', 'Gold'], dtype=object)]

In [19]:
print(encoded.shape)

(50000, 8)


In [20]:
np.hstack([encoded, df['n_disease'].values.reshape(50000, 1)])

array([[1., 2., 0., ..., 1., 0., 0.],
       [1., 2., 0., ..., 1., 0., 0.],
       [1., 3., 0., ..., 0., 0., 0.],
       ...,
       [1., 3., 1., ..., 2., 1., 1.],
       [1., 3., 1., ..., 1., 1., 1.],
       [0., 3., 1., ..., 0., 1., 1.]])

In [37]:
r_scores = r_regression(np.hstack([encoded, df[['n_disease', 'number_of_dependants', 'age']].values.reshape(50000, 3)]), df['annual_premium_amount'])

In [38]:
l = list(oe.get_feature_names_out())
l.extend(['n_disease', 'number_of_dependants', 'age'])
l

['gender',
 'region',
 'marital_status',
 'bmi_category',
 'smoking_status',
 'employment_status',
 'income_level',
 'insurance_plan',
 'n_disease',
 'number_of_dependants',
 'age']

In [39]:
d1 = {}
for x, y in zip(l, r_scores):
    d1[x] = round(y, 3)
d1

{'gender': 0.065,
 'region': 0.001,
 'marital_status': 0.517,
 'bmi_category': 0.333,
 'smoking_status': 0.229,
 'employment_status': 0.146,
 'income_level': 0.269,
 'insurance_plan': 0.834,
 'n_disease': 0.523,
 'number_of_dependants': 0.416,
 'age': 0.767}

In [48]:
d2 = dict(sorted(d1.items(), key=lambda x: x[1], reverse=True))
for k,v in d2.items():
    print(k, ":", v)

insurance_plan : 0.834
age : 0.767
n_disease : 0.523
marital_status : 0.517
number_of_dependants : 0.416
bmi_category : 0.333
income_level : 0.269
smoking_status : 0.229
employment_status : 0.146
gender : 0.065
region : 0.001


In [56]:
# let's also try the mutual info method for scoring
mir = mutual_info_regression(np.hstack([encoded, df[['n_disease', 'number_of_dependants', 'age']].values.reshape(50000, 3)]), df['annual_premium_amount'],
                             n_neighbors = 10)

In [59]:
d1 = {}
for x, y in zip(l, mir):
    d1[x] = round(y, 3)
d1

{'gender': 0.015,
 'region': 0.0,
 'marital_status': 0.169,
 'bmi_category': 0.149,
 'smoking_status': 0.144,
 'employment_status': 0.064,
 'income_level': 0.102,
 'insurance_plan': 0.616,
 'n_disease': 0.242,
 'number_of_dependants': 0.143,
 'age': 0.605}

In [62]:
d2 = dict(sorted(d1.items(), key=lambda x: x[1], reverse=True))
for k,v in d2.items():
    print(k, ":", v)

insurance_plan : 0.616
age : 0.605
n_disease : 0.242
marital_status : 0.169
bmi_category : 0.149
smoking_status : 0.144
number_of_dependants : 0.143
income_level : 0.102
employment_status : 0.064
gender : 0.015
region : 0.0


seems like gender and region have very low individual affect on predicting prem price

In [72]:
df = df.drop(['gender','region'], axis = 1)

In [80]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   age                    50000 non-null  int64 
 1   marital_status         50000 non-null  object
 2   number_of_dependants   50000 non-null  int64 
 3   bmi_category           50000 non-null  object
 4   smoking_status         50000 non-null  object
 5   employment_status      50000 non-null  object
 6   income_level           50000 non-null  object
 7   income_lakhs           50000 non-null  int64 
 8   insurance_plan         50000 non-null  object
 9   annual_premium_amount  50000 non-null  int64 
 10  n_disease              50000 non-null  int64 
dtypes: int64(5), object(6)
memory usage: 4.2+ MB


In [121]:
df.tail()

,age,marital_status,number_of_dependants,bmi_category,smoking_status,employment_status,income_level,income_lakhs,insurance_plan,annual_premium_amount,n_disease
49995,64,Unmarried,0,Normal,No Smoking,Self-Employed,<10L,3,Silver,20380,0
49996,54,Married,3,Obesity,Regular,Self-Employed,25L - 40L,28,Silver,25447,1
49997,42,Married,3,Normal,Occasional,Self-Employed,25L - 40L,33,Silver,16102,1
49998,55,Married,2,Overweight,Regular,Self-Employed,10L - 25L,16,Silver,23518,1
49999,48,Married,3,Normal,Occasional,Self-Employed,<10L,7,Silver,19730,1


In [153]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   age                    50000 non-null  int64 
 1   marital_status         50000 non-null  object
 2   number_of_dependants   50000 non-null  int64 
 3   bmi_category           50000 non-null  object
 4   smoking_status         50000 non-null  object
 5   employment_status      50000 non-null  object
 6   income_level           50000 non-null  object
 7   income_lakhs           50000 non-null  int64 
 8   insurance_plan         50000 non-null  object
 9   annual_premium_amount  50000 non-null  int64 
 10  n_disease              50000 non-null  int64 
dtypes: int64(5), object(6)
memory usage: 4.2+ MB


In [ ]:
# export the final data for modelling
df.to_excel("final.xlsx", index = False)